In [ ]:
#| default_exp measurement

In [ ]:
#| export
from __future__ import annotations

import hashlib
import json
import math
import numbers
import re
from datetime import datetime
from pathlib import Path

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

Speed and accuracy do not depend on the same things. How fast an engine runs depends on its **graph** — the
architecture, the widths, the numeric precision — and on the device and software stack; it does not depend on
the values of the weights. How accurate it is depends on the **weights** and on the evaluation set. So a
measurement is one of two kinds of row, and neither needs the other:

- a **latency row** times one engine on one device. It needs no accuracy, so a model with random weights
  (`weights='random'`) is measured like any other: this is how a latency predictor gets trained on
  architectures nobody has trained yet.
- a **quality row** measures one artifact's accuracy on one evaluation set, through the engine that runs it.

Rows are stored as JSON Lines, one object per line, appended and never rewritten; a file may hold both kinds.
`FIELDS[kind]` is the schema of each kind: every field's type, whether it is required, and the rules its
value obeys. A field that is not required may be absent or `None`, which means *not recorded*.

| Kind | Group | Fields |
|---|---|---|
| both | identity and provenance | `schema_version`, `kind`, `run_id`, `measured_at` (ISO-8601, UTC), `series`, `harness_commit`, `fasterai_version`, `report_url` |
| latency | graph | `architecture`, `graph_hash`, `input_resolution`, `macs`, `params`, `variant`, `recipe`, `weights` (`trained` or `random`), `model_id`, `model_revision` |
| latency | artifact | `precision`, `int8_mode` (iff `int8`), `engine_hash`, `size_mb`, `engine_layers`, `reformat_layers`, `layer_precisions` |
| latency | device and runtime | `device`, `device_host`, `runtime`, `runtime_version`, `platform_version`, `power_mode`, `clocks_locked`, `batch_size` |
| latency | conditions | `temp_start_c`, `temp_end_c`, `gpu_freq_mhz`, `throttled`, `warmup_s`, `duration_s`, `repeats`, `latency_spread_pct` |
| latency | latency | `latency_ms_p50`, `latency_ms_p90`, `latency_ms_p99`, `latency_scope`, `host_latency_ms`, `latency_source` |
| quality | model and artifact | `model_id`, `model_revision`, `model_hash`, `weights_license`, `engine_hash`, `precision`, `device` |
| quality | accuracy | `metric`, `eval_set`, `eval_set_hash`, `dataset_license`, `eval_n`, `accuracy`, `ci_low`, `ci_high`, `ci_method`, `measured_through`, `argmax_parity`, `parity_floor`, `baseline_run_id` |

- `series` is a free-text label grouping the rows measured in one session, typically one night; it is not part of any key.
- `graph_hash` is the sha256 of the ONNX graph with the values of every initializer removed, so two models of the same shape share it, whatever their weights. `graph_hash(onnx_path)` computes it.
- `latency_scope` says what the latency covers, e.g. engine execution only, or host transfers included; two latency rows are only comparable when it matches.
- `layer_precisions` counts the engine's layers by the precision they actually ran in, e.g. `{'int8': 52, 'fp16': 9}`.
- `engine_hash` is the sha256 of the engine file that ran; `model_hash` is whatever fingerprint the exporter records, so it is only a string.
- A quality row names the engine it was measured through; `engine_hash` may be `None` only for an accuracy measured on the development model.

Throughput is derived, never stored: `throughput(row)` is `batch_size * 1000 / latency_ms_p50` on a latency row.

Two questions are kept apart: `validate` asks whether a row is **well formed**, `status` whether the
measurement it records is **valid**. Neither compares a row to a latency or accuracy budget: a row carries no
verdict column, and whoever reads it recomputes their own. A well-formed row is `INVALID` when:

- latency: `latency_source` is not `measured`; `throttled` is true or not recorded; `latency_spread_pct` is
  above `max_spread_pct` (5 by default) or not recorded;
- quality: `measured_through` is not `deployed_engine`; `argmax_parity` is below `parity_floor` or not recorded.

`join(latency_rows, quality_rows)` pairs the VALID rows of both kinds that share an `engine_hash`: one dict per
pair, with the fields of both. The identity and provenance fields are kept per side, as `latency_run_id`,
`quality_run_id`, `latency_measured_at` and so on; `schema_version` is kept once. The fields both kinds carry
(`engine_hash`, `precision`, `device`, `model_id`, `model_revision`) are one column: a side that did not
record one takes the other's value, and two sides that disagree on the same engine raise.

In [ ]:
#| export
SCHEMA_VERSION = 1
_UTC = r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d+)?(Z|\+00:00)'
_HEX = r'[0-9a-f]{64}'
_PCT = dict(ge=0, le=100)
_PRECISION = dict(type=str, required=True, choices=('fp32', 'fp16', 'int8', 'mixed'))

# name -> type, required, and the declarative rules its value obeys; a field that is not required may be absent or None
_SHARED = {
    'schema_version': dict(type=int, required=True, choices=(SCHEMA_VERSION,)),
    'kind': dict(type=str, required=True, choices=('latency', 'quality')),
    'run_id': dict(type=str, required=True),
    'measured_at': dict(type=str, required=True, pattern=_UTC, date=True),
    'series': dict(type=str, required=False),
    'harness_commit': dict(type=str, required=True),
    'fasterai_version': dict(type=str, required=False),
    'report_url': dict(type=str, required=False),
}

_LATENCY = {
    # graph
    'architecture': dict(type=str, required=True),
    'graph_hash': dict(type=str, required=True, pattern=_HEX),   # `graph_hash`: the ONNX graph without its weight values
    'input_resolution': dict(type=int, required=True, gt=0),
    'macs': dict(type=int, required=True, gt=0),
    'params': dict(type=int, required=True, gt=0),
    'variant': dict(type=str, required=True),
    'recipe': dict(type=dict, required=True, json=True),
    'weights': dict(type=str, required=True, choices=('trained', 'random')),
    'model_id': dict(type=str, required=False),
    'model_revision': dict(type=str, required=False),
    # artifact
    'precision': _PRECISION,
    'int8_mode': dict(type=str, required=False, choices=('implicit', 'explicit')),
    'engine_hash': dict(type=str, required=True, pattern=_HEX),   # sha256 of the engine file that ran
    'size_mb': dict(type=float, required=True, gt=0),
    'engine_layers': dict(type=int, required=False, gt=0),
    'reformat_layers': dict(type=int, required=False, ge=0),
    'layer_precisions': dict(type=dict, required=False, counts=True),
    # device and runtime
    'device': dict(type=str, required=True),
    'device_host': dict(type=str, required=True, choices=('own', 'rented', 'partner')),
    'runtime': dict(type=str, required=True, choices=('tensorrt', 'openvino', 'onnxruntime', 'hailo', 'axelera')),
    'runtime_version': dict(type=str, required=True),
    'platform_version': dict(type=str, required=False),
    'power_mode': dict(type=str, required=False),
    'clocks_locked': dict(type=bool, required=False),
    'batch_size': dict(type=int, required=True, ge=1),
    # conditions
    'temp_start_c': dict(type=float, required=False),
    'temp_end_c': dict(type=float, required=False),
    'gpu_freq_mhz': dict(type=float, required=False, gt=0),
    'throttled': dict(type=bool, required=False),
    'warmup_s': dict(type=float, required=False, ge=0),
    'duration_s': dict(type=float, required=False, ge=0),
    'repeats': dict(type=int, required=True, ge=1),
    'latency_spread_pct': dict(type=float, required=False, ge=0),
    # latency
    'latency_ms_p50': dict(type=float, required=True, gt=0),
    'latency_ms_p90': dict(type=float, required=False, gt=0),
    'latency_ms_p99': dict(type=float, required=False, gt=0),
    'latency_scope': dict(type=str, required=True),
    'host_latency_ms': dict(type=float, required=False, gt=0),
    'latency_source': dict(type=str, required=True, choices=('measured', 'estimated')),
}

_QUALITY = {
    'model_id': dict(type=str, required=True),
    'model_revision': dict(type=str, required=True),
    'model_hash': dict(type=str, required=False),   # whatever fingerprint the exporter records, so not pinned
    'weights_license': dict(type=str, required=True),
    'engine_hash': dict(type=str, required=False, pattern=_HEX),   # the engine the accuracy was measured through
    'precision': _PRECISION,
    'device': dict(type=str, required=False),
    'metric': dict(type=str, required=True),
    'eval_set': dict(type=str, required=True),
    'eval_set_hash': dict(type=str, required=True),
    'dataset_license': dict(type=str, required=True),
    'eval_n': dict(type=int, required=True, gt=0),
    'accuracy': dict(type=float, required=True, **_PCT),
    'ci_low': dict(type=float, required=True, **_PCT),
    'ci_high': dict(type=float, required=True, **_PCT),
    'ci_method': dict(type=str, required=True, choices=('wilson', 'bootstrap')),
    'measured_through': dict(type=str, required=True, choices=('deployed_engine', 'dev_model')),
    'argmax_parity': dict(type=float, required=False, **_PCT),
    'parity_floor': dict(type=float, required=True, **_PCT),
    'baseline_run_id': dict(type=str, required=False),
}

FIELDS = {'latency': {**_SHARED, **_LATENCY}, 'quality': {**_SHARED, **_QUALITY}}


def _item(o):
    "A numpy scalar as the Python number JSON knows"
    if isinstance(o, numbers.Number) and hasattr(o, 'item'): return o.item()
    raise TypeError(f'{type(o).__name__} is not JSON serializable')


def _json(obj, **kw): return json.dumps(obj, default=_item, allow_nan=False, **kw)
def _canonical(d): return _json(d, sort_keys=True, separators=(',', ':'))


def _is_json(v):
    try: return _json(v) is not None
    except (TypeError, ValueError): return False


def _is_date(v):
    try: return datetime.fromisoformat(v.replace('Z', '+00:00')) is not None   # Python 3.10 does not read 'Z'
    except ValueError: return False


def _typed(v, t):
    "`v` is a `t`; a bool is not a number, a numpy integer is an int, an int is a float, a float is finite"
    if isinstance(v, bool): return t is bool
    if t is int: return isinstance(v, numbers.Integral)
    if t is float: return isinstance(v, numbers.Real) and math.isfinite(v)
    return isinstance(v, t)


_RULES = {
    'choices': (lambda v, c: v in c, 'not one of {}'),
    'gt': (lambda v, b: v > b, 'not > {}'),
    'ge': (lambda v, b: v >= b, 'not >= {}'),
    'le': (lambda v, b: v <= b, 'not <= {}'),
    'pattern': (lambda v, p: re.fullmatch(p, v) is not None, 'does not match {}'),
    'date': (lambda v, _: _is_date(v), 'is not a date'),
    'json': (lambda v, _: _is_json(v), 'is not JSON'),
    'counts': (lambda v, _: all(isinstance(k, str) and _typed(n, int) and n >= 0 for k, n in v.items()),
               'is not a {{name: count}} dict'),
}
_ORDERED = {'latency': (('latency_ms_p50', 'latency_ms_p90', 'latency_ms_p99'), ('reformat_layers', 'engine_layers')),
            'quality': (('ci_low', 'accuracy', 'ci_high'),)}


def _field_problems(name, spec, v):
    "The problems of one field value, against its spec"
    if v is None: return [f'{name}: missing'] if spec['required'] else []
    if not _typed(v, spec['type']): return [f'{name}: expected {spec["type"].__name__}, got {v!r}']
    return [f'{name}: {v!r} {msg.format(spec[r])}' for r, (ok, msg) in _RULES.items() if r in spec and not ok(v, spec[r])]


def _cross_problems(row):
    "Problems between fields; each field involved is already well typed"
    out, kind = [], row['kind']
    if kind == 'latency':
        p, mode = row['precision'], row.get('int8_mode')
        if p == 'int8' and mode is None: out.append('int8_mode: missing, required when precision is int8')
        if p != 'int8' and mode is not None: out.append(f'int8_mode: {mode!r} given for precision {p!r}')
        if row['weights'] == 'random' and row.get('model_id') is not None:
            out.append(f"model_id: {row['model_id']!r} given for random weights")
    elif row['measured_through'] == 'deployed_engine' and row.get('engine_hash') is None:
        out.append('engine_hash: missing, required when measured through the deployed engine')
    for fields in _ORDERED[kind]:
        vals = [(f, row[f]) for f in fields if row.get(f) is not None]
        if [v for _, v in vals] != sorted(v for _, v in vals):
            out.append(f"{', '.join(fields)}: not in order, " + ', '.join(f'{f}={v}' for f, v in vals))
    return out


def validate(
    row: dict,  # one measurement row, of either kind
) -> list[str]:
    "Every structural problem of `row`, each prefixed by its field name; an empty list means well formed"
    if not isinstance(row, dict): return [f'row: expected a dict, got {type(row).__name__}']
    kind = row.get('kind')
    if kind is None: return ['kind: missing']
    if not isinstance(kind, str) or kind not in FIELDS: return [f'kind: {kind!r} not one of {tuple(FIELDS)}']
    fields = FIELDS[kind]
    out = [f'{k}: unknown field for a {kind} row' for k in row if k not in fields]
    for name, spec in fields.items(): out += _field_problems(name, spec, row.get(name))
    return out or _cross_problems(row)

In [ ]:
show_doc(validate)

In [ ]:
#| export
# per kind: field, when its value makes the measurement invalid, and why; a field that is None is not recorded
_VALIDITY = {
    'latency': (
        ('latency_source', lambda v, r, m: v != 'measured', 'latency not measured on the device'),
        ('throttled', lambda v, r, m: v, 'the device throttled during the session'),
        ('latency_spread_pct', lambda v, r, m: v > m, '{v} above {m}'),
    ),
    'quality': (
        ('measured_through', lambda v, r, m: v != 'deployed_engine', 'accuracy not measured through the deployed engine'),
        ('argmax_parity', lambda v, r, m: v < r['parity_floor'], '{v} below parity_floor {r[parity_floor]}'),
    ),
}


def status(
    row: dict,                    # one measurement row, of either kind
    max_spread_pct: float = 5.0,  # largest (max-min)/median latency spread over repeats, in percent
) -> tuple[str, list[str]]:
    "('VALID', []) if `row` is a usable measurement, else ('INVALID', reasons); not a verdict against any budget"
    reasons = validate(row)
    if reasons: return 'INVALID', reasons
    for f, bad, why in _VALIDITY[row['kind']]:
        v = row.get(f)
        if v is None: reasons.append(f'{f}: not recorded')
        elif bad(v, row, max_spread_pct): reasons.append(f'{f}: ' + why.format(v=v, r=row, m=max_spread_pct))
    return ('INVALID', reasons) if reasons else ('VALID', [])

In [ ]:
show_doc(status)

In [ ]:
#| export
_KEYS = {
    'latency': ('graph_hash', 'weights', 'precision', 'int8_mode', 'batch_size', 'device', 'runtime',
                'runtime_version', 'platform_version', 'measured_at'),
    'quality': ('model_id', 'model_revision', 'engine_hash', 'eval_set_hash', 'metric', 'measured_at'),
}


def key(
    row: dict,  # one well-formed measurement row
) -> tuple:
    "The identity a row is unique on, its kind first; a software update on the device gives a new latency key"
    return (row['kind'],) + tuple(row.get(f) for f in _KEYS[row['kind']])


def throughput(
    row: dict,  # one latency row
) -> float:
    "Images per second at the timed batch, derived from the median latency and never stored"
    if row.get('kind') != 'latency': raise ValueError(f"throughput needs a latency row, got kind {row.get('kind')!r}")
    return row['batch_size'] * 1000 / row['latency_ms_p50']

In [ ]:
show_doc(key)

In [ ]:
show_doc(throughput)

A latency row is unique on its graph, its weights (`trained` or `random`), precision, batch size,
device, software stack and date; a quality row on its model, revision, engine, evaluation set, metric and
date. Uniqueness is checked per kind.

In [ ]:
#| export
_OWN = tuple(f for f in _SHARED if f not in ('schema_version', 'kind'))   # kept per side, prefixed
_BOTH = tuple(f for f in _LATENCY if f in _QUALITY)                        # one column, both sides must agree


def _pair(lat, qual):
    "One joined dict from a latency row and a quality row of the same engine"
    out = {'schema_version': lat['schema_version']}
    for side, r in (('latency', lat), ('quality', qual)):
        out |= {f'{side}_{f}': r.get(f) for f in _OWN}
        out |= {f: r.get(f) for f in FIELDS[side] if f not in _SHARED and f not in _BOTH}
    for f in _BOTH:
        a, b = lat.get(f), qual.get(f)
        if None not in (a, b) and a != b:
            raise ValueError(f"{f}: latency row {lat['run_id']} says {a!r}, quality row {qual['run_id']} {b!r}, same engine")
        out[f] = b if a is None else a
    return out


def join(
    latency_rows: list[dict],  # latency rows; rows of the other kind and invalid rows are skipped
    quality_rows: list[dict],  # quality rows; rows of the other kind and invalid rows are skipped
) -> list[dict]:
    "One dict per pair of VALID latency and quality rows sharing an `engine_hash`"
    by_engine = {}
    for q in quality_rows:
        if q.get('kind') == 'quality' and status(q)[0] == 'VALID': by_engine.setdefault(q['engine_hash'], []).append(q)
    return [_pair(l, q) for l in latency_rows if l.get('kind') == 'latency' and status(l)[0] == 'VALID'
            for q in by_engine.get(l['engine_hash'], [])]

In [ ]:
show_doc(join)

In [ ]:
#| export
def graph_hash(
    onnx_path: str | Path,  # an ONNX file
) -> str:
    "sha256 of the ONNX graph with every initializer's values removed and its name, shape and type kept"
    try: import onnx
    except ImportError: raise ImportError("graph_hash needs onnx: pip install 'fastermodels[onnx]'") from None
    m = onnx.load(str(onnx_path), load_external_data=False)
    # ponytail: top-level initializers only; Constant nodes and subgraph initializers keep their values
    for t in m.graph.initializer:
        name, dtype, dims = t.name, t.data_type, list(t.dims)
        t.Clear()
        t.name, t.data_type = name, dtype
        t.dims.extend(dims)
    h = hashlib.sha256(m.graph.SerializeToString(deterministic=True))
    for o in sorted((o.domain, o.version) for o in m.opset_import): h.update(f'{o[0]}:{o[1]};'.encode())
    return h.hexdigest()

In [ ]:
show_doc(graph_hash)

Initializer names are part of the graph: renaming a weight gives a new hash, as it does to the
engine a runtime builds from it. Opset versions are hashed too. `graph_hash` needs the optional `onnx`
extra: `pip install 'fastermodels[onnx]'`.

In [ ]:
#| export
def _parse(text, path):
    "The rows of a JSON Lines text; a malformed line raises with its number"
    rows = []
    for i, line in enumerate(text.splitlines(), 1):
        if not line.strip(): continue
        try: r = json.loads(line)
        except json.JSONDecodeError as e: raise ValueError(f'{path}:{i}: not JSON ({e.msg})') from None
        if not isinstance(r, dict): raise ValueError(f'{path}:{i}: not a JSON object')
        rows.append(r)
    return rows


def read_rows(
    path: str | Path,  # a JSON Lines file of measurement rows, of either kind
) -> list[dict]:
    "The rows of `path`, as stored; `status` judges them"
    return _parse(Path(path).read_text(), path)


def write_row(
    path: str | Path,  # a JSON Lines file, created if absent
    row: dict,         # the measurement row to append
):
    "Append `row` to `path`; refuses a malformed row and a row whose key is already in the file"
    problems = validate(row)
    if problems: raise ValueError('row refused: ' + '; '.join(problems))
    p, k = Path(path), key(row)
    text = p.read_text() if p.exists() else ''
    if k in {key(r) for r in _parse(text, path) if r.get('kind') == row['kind']}:
        raise ValueError(f'row refused: key {k} already in {path}')
    # ponytail: no file lock, one writer at a time
    with p.open('a') as f: f.write(('\n' if text and not text.endswith('\n') else '') + _json(row) + '\n')

In [ ]:
show_doc(read_rows)

In [ ]:
show_doc(write_row)

In [ ]:
#| export
def to_parquet(
    rows: list[dict],  # well-formed measurement rows of one kind, e.g. from `read_rows`
    path: str | Path,  # the Parquet file to write
):
    "Write `rows` as one Parquet table, one column per field of their kind, dict fields as canonical JSON"
    try:
        import pandas as pd
        import pyarrow  # noqa: F401
    except ImportError: raise ImportError("to_parquet needs pandas and pyarrow: pip install 'fastermodels[parquet]'") from None
    kinds = sorted({str(r.get('kind')) for r in rows})
    if len(kinds) != 1: raise ValueError(f'to_parquet writes one kind per file, got {kinds or "no rows"}')
    for i, r in enumerate(rows):
        problems = validate(r)
        if problems: raise ValueError(f'row {i} refused: ' + '; '.join(problems))
    fields = FIELDS[kinds[0]]
    table = [{k: _canonical(r[k]) if fields[k]['type'] is dict and r.get(k) is not None else r.get(k) for k in fields}
             for r in rows]
    pd.DataFrame(table, columns=list(fields)).to_parquet(path, engine='pyarrow', index=False)

In [ ]:
show_doc(to_parquet)

`to_parquet` writes one kind per file and needs the optional `parquet` extra:
`pip install 'fastermodels[parquet]'`.

---

## Usage

A latency row, here for an engine built from a model with random weights:

```python
from fastermodels import measurement

lat = dict(schema_version=1, kind='latency', run_id='2026-09-28-a', measured_at='2026-09-28T10:00:00Z',
           harness_commit='deadbee', architecture='resnet50', graph_hash=measurement.graph_hash('model.onnx'),
           input_resolution=224, macs=2_900_000_000, params=17_000_000, variant='p30', recipe={'prune': 0.30},
           weights='random', precision='fp16', engine_hash='0' * 64, size_mb=34.1,
           layer_precisions={'fp16': 61}, device='jetson-nano-4gb', device_host='own', runtime='tensorrt',
           runtime_version='8.2.1', platform_version='jetpack-4.6.1', batch_size=1, throttled=False,
           repeats=5, latency_spread_pct=1.2, latency_ms_p50=2.1,
           latency_scope='engine execute, host transfers excluded', latency_source='measured')

measurement.validate(lat)                 # [] : well formed
measurement.status(lat)                   # ('VALID', [])
measurement.throughput(lat)               # 476.19... images per second, derived
measurement.write_row('rows.jsonl', lat)  # appends one line; a second call with the same row is refused
```

A quality row, and the join of the two kinds on the engine they share:

```python
qual = dict(schema_version=1, kind='quality', run_id='2026-09-28-b', measured_at='2026-09-28T11:00:00Z',
            harness_commit='deadbee', model_id='org/resnet50-p30', model_revision='a1b2c3d',
            weights_license='apache-2.0', engine_hash='1' * 64, precision='fp16', device='jetson-nano-4gb',
            metric='top1', eval_set='imagenet-val', eval_set_hash='sha256:...', dataset_license='custom',
            eval_n=50_000, accuracy=78.9, ci_low=78.5, ci_high=79.3, ci_method='wilson',
            measured_through='deployed_engine', argmax_parity=99.7, parity_floor=99.0)

rows = measurement.read_rows('rows.jsonl')
measurement.join(rows, rows)              # each kind is picked from the list it belongs to
measurement.to_parquet([r for r in rows if r['kind'] == 'latency'], 'latency.parquet')
```

---

## See Also

- [Model](00_model.html) - the published model a quality row measures
- [Eval](01_eval.html) - `wilson`, the interval `ci_method='wilson'` refers to, and `agreement`
- [Card](02_card.html) - where a published latency is written down
- [Gate](03_gate.html) - the conditions an artifact meets before it is published

Tests live in `nbs/tests/test_measurement.ipynb`.